# 01_download: 数据下载

**学生**：谢婧怡 25210094

说明：使用 akshare 下载个股、指数、宏观与财务数据的模板。运行本单元将把原始 CSV 保存到 `dshw-p01/data/`，并在 `dshw-p01/download_log.txt` 中记录每次下载状态。

In [3]:

import os
import time
from datetime import datetime
import pandas as pd
import akshare as ak

# 配置
stocks = ['601398','601988','600104','000625','000002','600519','000858','600028','600050','002352']
start_date = '2020-01-01'
end_date = datetime.today().strftime('%Y-%m-%d')
base_dir = '.'
data_dirs = {
    'stock': os.path.join(base_dir, 'data', 'stock'),
    'index': os.path.join(base_dir, 'data', 'index'),
    'macro': os.path.join(base_dir, 'data', 'macro'),
    'finance': os.path.join(base_dir, 'data', 'finance'),
    'clean': os.path.join(base_dir, 'data', 'clean'),
    'combined': os.path.join(base_dir, 'data', 'combined'),
}
for d in data_dirs.values():
    os.makedirs(d, exist_ok=True)
os.makedirs(os.path.join(base_dir, 'output'), exist_ok=True)
log_path = os.path.join(base_dir, 'download_log.txt')

def write_log(status, name, info=''):
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{ts}] {status:7s} {name:12s} {info}\n'
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(line)
    print(line, end='')

def market_prefix(code):
    return 'sh' if str(code).startswith('6') else 'sz'

# 下载个股日线（后复权）
for code in stocks:
    name = f'stock_{code}'
    symbol = market_prefix(code) + code
    try:
        # akshare 的接口版本可能不同，优先尝试常用函数
        try:
            df = ak.stock_zh_a_daily(symbol=symbol, start_date=start_date, end_date=end_date, adjust='hfq')
        except Exception:
            df = ak.stock_zh_a_hist(symbol=symbol, start_date=start_date, end_date=end_date, adjust='qfq')
        if df is None or df.empty:
            write_log('FAILED', name, 'No data returned')
            continue
        # 保存为 CSV
        out_path = os.path.join(data_dirs['stock'], f'stock_{code}.csv')
        df.to_csv(out_path, index=False, encoding='utf-8-sig')
        write_log('SUCCESS', name, f'shape={df.shape}')
        time.sleep(1)
    except Exception as e:
        write_log('FAILED', name, f'Error: {e}')


# 下载沪深300 指数（日线）—— 使用 baostock，路径和日志统一
index_code = '000300'
index_name = f'index_{index_code}'
try:
    import baostock as bs
    # 登录
    lg = bs.login()
    if lg.error_code != '0':
        write_log('FAILED', index_name, f'baostock login failed: {lg.error_msg}')
    else:
        # 获取沪深300指数日K线（代码 sh.000300）
        rs = bs.query_history_k_data_plus(
            "sh.000300",
            "date,open,high,low,close,volume,amount",
            start_date=start_date,
            end_date=end_date,
            frequency="d",
            adjustflag="2"   # 2=前复权，指数无影响
        )
        df_index = rs.get_data()
        bs.logout()
        
        if df_index.empty:
            write_log('FAILED', index_name, 'No data returned')
        else:
            # 转换日期格式
            df_index['date'] = pd.to_datetime(df_index['date'])
            # 保存到正确的目录
            out_path = os.path.join(data_dirs['index'], f'index_{index_code}.csv')
            df_index.to_csv(out_path, index=False, encoding='utf-8-sig')
            write_log('SUCCESS', index_name, f'shape={df_index.shape}')
            print(df_index.head())
except Exception as e:
    write_log('FAILED', index_name, f'Error: {e}')

# 下载第二个指数（示例：中证500 — 000905）
try:
    idx2_name = 'index_000905'
    idx2_df = None
    last_err = None
    # 尝试通过成分接口获取（如果可用）
    try:
        if hasattr(ak, 'index_stock_cons_sina'):
            zz500 = ak.index_stock_cons_sina("000905")
            if zz500 is not None and not zz500.empty:
                idx2_df = zz500
                write_log('SUCCESS', idx2_name, f'use index_stock_cons_sina shape={idx2_df.shape}')
    except Exception as e:
        last_err = e
    # 回退到常见的指数接口（带重试）
    if idx2_df is None or idx2_df.empty:
        cand_fns = ['index_zh_a_hist', 'index_zh_a_daily', 'index_zh_a_index_daily', 'index_zh_a_spot']
        symbol_variants = ['000905', '000905.SH']
        for sym in symbol_variants:
            for fn in cand_fns:
                if not hasattr(ak, fn):
                    continue
                func = getattr(ak, fn)
                for attempt in range(1, 6):
                    try:
                        idx2_df = func(symbol=sym, start_date=start_date, end_date=end_date)
                        if idx2_df is not None and not idx2_df.empty:
                            break
                    except Exception as e:
                        last_err = e
                        time.sleep(min(2 * attempt, 10))
                if idx2_df is not None and not idx2_df.empty:
                    break
            if idx2_df is not None and not idx2_df.empty:
                break
    # 结果处理
    if idx2_df is None or idx2_df.empty:
        err_msg = f'No data returned. Last error: {repr(last_err)}' if last_err is not None else 'No data returned'
        write_log('FAILED', idx2_name, err_msg)
    else:
        try:
            if 'date' in idx2_df.columns:
                idx2_df['date'] = pd.to_datetime(idx2_df['date'], errors='coerce')
        except Exception:
            pass
        idx2_path = os.path.join(data_dirs['index'], 'index_000905.csv')
        idx2_df.to_csv(idx2_path, index=False, encoding='utf-8-sig')
        write_log('SUCCESS', idx2_name, f'shape={idx2_df.shape}')
except Exception as e:
    write_log('FAILED', 'index_000905', f'Error: {e}')

# 下载 CPI（优先尝试同比/年度数据）
try:
    macro_name = 'macro_cpi'
    cpi = None
    last_err = None
    cpi_candidates = ['macro_china_cpi_yearly', 'macro_china_cpi_monthly', 'macro_china_cpi']
    for fn in cpi_candidates:
        if not hasattr(ak, fn):
            continue
        try:
            cpi = getattr(ak, fn)()
            if cpi is not None and not cpi.empty:
                break
        except Exception as e:
            last_err = e
            time.sleep(1)
    if cpi is None or cpi.empty:
        err_msg = f'No data returned. Last error: {repr(last_err)}' if last_err is not None else 'No data returned'
        write_log('FAILED', macro_name, err_msg)
    else:
        cpi_path = os.path.join(data_dirs['macro'], 'macro_cpi.csv')
        cpi.to_csv(cpi_path, index=False, encoding='utf-8-sig')
        write_log('SUCCESS', macro_name, f'shape={cpi.shape}')
except Exception as e:
    write_log('FAILED', 'macro_cpi', f'Error: {e}')

# 下载 M2（优先尝试同比/年度数据）
try:
    macro_name = 'macro_m2'
    m2 = None
    last_err = None
    m2_candidates = ['macro_china_m2_yearly', 'macro_china_m2_monthly', 'macro_china_money_supply_yearly', 'macro_china_money_supply_monthly', 'macro_china_money_supply']
    for fn in m2_candidates:
        if not hasattr(ak, fn):
            continue
        try:
            m2 = getattr(ak, fn)()
            if m2 is not None and not m2.empty:
                break
        except Exception as e:
            last_err = e
            time.sleep(1)
    if m2 is None or m2.empty:
        err_msg = f'No data returned. Last error: {repr(last_err)}' if last_err is not None else 'No data returned'
        write_log('FAILED', macro_name, err_msg)
    else:
        m2_path = os.path.join(data_dirs['macro'], 'macro_m2.csv')
        m2.to_csv(m2_path, index=False, encoding='utf-8-sig')
        write_log('SUCCESS', macro_name, f'shape={m2.shape}')
except Exception as e:
    write_log('FAILED', 'macro_m2', f'Error: {e}')

print('下载完成。请检查 dshw-p01/download_log.txt 以查看详细记录。')


[2026-04-03 19:31:08] SUCCESS stock_601398 shape=(1514, 9)
[2026-04-03 19:31:10] SUCCESS stock_601988 shape=(1514, 9)
[2026-04-03 19:31:13] SUCCESS stock_600104 shape=(1514, 9)
[2026-04-03 19:31:16] SUCCESS stock_000625 shape=(1514, 9)
[2026-04-03 19:31:19] SUCCESS stock_000002 shape=(1514, 9)
[2026-04-03 19:31:21] SUCCESS stock_600519 shape=(1514, 9)
[2026-04-03 19:31:24] SUCCESS stock_000858 shape=(1514, 9)
[2026-04-03 19:31:27] SUCCESS stock_600028 shape=(1514, 9)
[2026-04-03 19:31:30] SUCCESS stock_600050 shape=(1514, 9)
[2026-04-03 19:31:32] SUCCESS stock_002352 shape=(1511, 9)
[2026-04-03 19:31:33] FAILED  index_000300 Error: No module named 'baostock'
[2026-04-03 19:31:35] SUCCESS index_000905 use index_stock_cons_sina shape=(500, 15)
[2026-04-03 19:31:35] SUCCESS index_000905 shape=(500, 15)
[2026-04-03 19:31:46] SUCCESS macro_cpi    shape=(477, 5)
[2026-04-03 19:31:56] SUCCESS macro_m2     shape=(395, 5)
下载完成。请检查 dshw-p01/download_log.txt 以查看详细记录。
